# Connect to S3 and read a PDF

This notebook connects to the **mmx-amazon-s3-bucket** S3 bucket using the dedicated
`s3-bucket` AWS CLI profile (an IAM user access key, **not** your root account),
lists the objects, downloads `Fabric Data Agent.pdf`, and reads its text.

> Credentials come from the `s3-bucket` profile in `~/.aws/credentials`, so no keys are stored in this notebook.


## 1. Imports and configuration


In [1]:
import boto3
from pathlib import Path

PROFILE   = 's3-bucket'
BUCKET    = 'mmx-amazon-s3-bucket'
PDF_KEY   = 'Fabric Data Agent.pdf'
DOWNLOAD_DIR = Path('downloads')
DOWNLOAD_DIR.mkdir(exist_ok=True)


## 2. Create an S3 client from the profile


In [2]:
session = boto3.Session(profile_name=PROFILE)
s3 = session.client('s3')
print('Connected as region:', session.region_name)


Connected as region: us-east-1


## 3. List objects in the bucket


In [3]:
resp = s3.list_objects_v2(Bucket=BUCKET)
for obj in resp.get('Contents', []):
    print(f"{obj['Key']}  ({obj['Size']:,} bytes)")


Fabric Data Agent.pdf  (649,958 bytes)


## 4. Download the PDF


In [4]:
local_path = DOWNLOAD_DIR / PDF_KEY
s3.download_file(BUCKET, PDF_KEY, str(local_path))
print('Downloaded to:', local_path.resolve())
print('Size on disk:', local_path.stat().st_size, 'bytes')


Downloaded to: C:\Users\memasanz\repos\aws-connect\downloads\Fabric Data Agent.pdf
Size on disk: 649958 bytes


## 5. Read text from the PDF


In [5]:
from pypdf import PdfReader

reader = PdfReader(str(local_path))
print('Number of pages:', len(reader.pages))

first_page_text = reader.pages[0].extract_text() or '(no extractable text on page 1)'
print('\n--- Page 1 text (first 1000 chars) ---\n')
print(first_page_text[:1000])


Number of pages: 13

--- Page 1 text (first 1000 chars) ---

Microsoft Fabric
Microsoft 
Fabric Data 
Agent
Prepare your data for AI innovation


## 5b. S3 object metadata

`head_object` returns metadata about the file **without downloading it**.


In [6]:
head = s3.head_object(Bucket=BUCKET, Key=PDF_KEY)

print('Size (bytes):   ', f"{head['ContentLength']:,}")
print('Content type:   ', head.get('ContentType'))
print('Last modified:  ', head['LastModified'])
print('ETag:           ', head['ETag'].strip('"'))
print('Storage class:  ', head.get('StorageClass', 'STANDARD'))
print('Encryption:     ', head.get('ServerSideEncryption', 'none'))

# Any custom user-defined metadata (x-amz-meta-*)
user_meta = head.get('Metadata', {})
print('User metadata:  ', user_meta if user_meta else '(none)')


Size (bytes):    649,958
Content type:    application/pdf
Last modified:   2026-08-07 19:22:38+00:00
ETag:            ce67bdcdd934f613eeead5b592bba0ac
Storage class:   STANDARD
Encryption:      AES256
User metadata:   (none)


## 5c. PDF document metadata

Metadata embedded inside the PDF itself (author, title, producer, dates).


In [7]:
info = reader.metadata
if info:
    print('Title:        ', info.title)
    print('Author:       ', info.author)
    print('Subject:      ', info.subject)
    print('Creator:      ', info.creator)
    print('Producer:     ', info.producer)
    print('Created:      ', info.creation_date)
    print('Modified:     ', info.modification_date)
else:
    print('No embedded PDF metadata found.')

print('Page count:   ', len(reader.pages))
w = reader.pages[0].mediabox.width
h = reader.pages[0].mediabox.height
print(f'Page 1 size:   {float(w):.0f} x {float(h):.0f} pts')


Title:         None
Author:        
Subject:       None
Creator:       Microsoft® PowerPoint® for Microsoft 365
Producer:      Microsoft® PowerPoint® for Microsoft 365
Created:       2025-10-06 08:36:07-05:00
Modified:      2025-10-06 08:36:07-05:00
Page count:    13
Page 1 size:   960 x 540 pts


## 6. (Optional) Upload a file back to the bucket

Uncomment to test write access.


In [8]:
# note = DOWNLOAD_DIR / 'hello.txt'
# note.write_text('uploaded from the notebook')
# s3.upload_file(str(note), BUCKET, 'hello.txt')
# print('Uploaded hello.txt')
